# Edge-Enhanced GraphSAGE — Live Demo

**Component:** Member 1 — Relational Fraud Detector
**Trained model:** Stage 3a (Edge-MLP + Focal Loss) — the best-performing ablation stage
**Test F1:** 0.5387  |  **AUROC:** 0.9497  |  **Recall:** 0.5663

This notebook runs **live inference** on the PaySim graph: picks a real fraud transaction,
runs the trained GraphSAGE model, extracts the k=2 hop suspicious subgraph,
visualizes the mule ring, and builds the forensic JSON payload sent to the Fusion Engine.


## 1. Setup


In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import k_hop_subgraph

from graphsage.models.edge_sage import EdgeEnhancedGraphSAGE
from graphsage.training.threshold_tuning import metrics_at_threshold, find_best_threshold_for_f1

from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
GRAPH_PATH = REPO / 'data' / 'graph' / 'paysim_graph.pt'
CKPT_PATH = REPO / 'checkpoints' / 'stage3a_focal.pt'
PARQUET_PATH = REPO / 'data' / 'processed' / 'features.parquet'

print('PyTorch:', torch.__version__)
print('Graph file:', GRAPH_PATH.name, GRAPH_PATH.stat().st_size // 1024**2, 'MB')
print('Checkpoint:', CKPT_PATH.name)


## 2. Load the trained graph and Stage 3a checkpoint


In [ ]:
# Load the full PaySim graph (3.27M nodes, 2.77M edges)
data = torch.load(GRAPH_PATH, weights_only=False)
print(data)
print(f'Total mules in graph: {int(data.y.sum().item()):,}')
print(f'Test mules (held out): {int((data.test_mask & data.y.bool()).sum().item()):,}')


In [ ]:
# Load the Stage 3a model
ckpt = torch.load(CKPT_PATH, weights_only=False, map_location='cpu')
hp = ckpt['hyperparameters']

model = EdgeEnhancedGraphSAGE(
    in_dim=int(data.x.shape[1]),
    edge_dim=int(data.edge_attr.shape[1]),
    hidden_dim=hp.get('hidden_dim', 64),
    edge_mlp_hidden=hp.get('edge_mlp_hidden', 32),
    dropout=hp.get('dropout', 0.3),
)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f'Model loaded - {sum(p.numel() for p in model.parameters()):,} parameters')
print(f'  Loss class: {ckpt.get("loss_class", "unknown")}')
print(f'  Best epoch: {ckpt.get("best_epoch", "?")}')


## 3. Run inference + threshold tuning


In [ ]:
# Forward pass on the full graph (CPU is fine - tiny model, ~20 sec)
with torch.no_grad():
    logits, attentions = model.forward_with_attention(
        data.x, data.edge_index, data.edge_attr
    )
    probs = torch.sigmoid(logits)

# Tune threshold on validation set
best_thresh, val_f1 = find_best_threshold_for_f1(
    logits[data.val_mask], data.y[data.val_mask]
)
print(f'Best threshold from validation: {best_thresh:.4f}  (val F1 = {val_f1:.4f})')

# Test set metrics at this threshold
test_m = metrics_at_threshold(logits[data.test_mask], data.y[data.test_mask], best_thresh)
print(f'\nTest set metrics at tuned threshold:')
for k, v in test_m.items():
    print(f'  {k:12s}: {v:.4f}')


## 4. Pick a high-confidence fraud prediction

Choose a test-set node where the model predicts fraud with high confidence and the ground truth confirms it.


In [ ]:
# Find test nodes that are mules AND model predicts as fraud
test_mules = (data.test_mask & data.y.bool()).nonzero(as_tuple=True)[0]
test_mule_probs = probs[test_mules]

# Pick the highest-confidence true mule
top_idx = test_mules[test_mule_probs.argmax()].item()
top_prob = float(probs[top_idx].item())

print(f'Selected node: {top_idx}')
print(f'  Predicted fraud probability: {top_prob:.4f}')
print(f'  Ground truth: MULE (received fraud)')
print(f'  Above tuned threshold ({best_thresh:.4f}): {top_prob >= best_thresh}')


## 5. Extract the k=2 hop suspicious subgraph (Novelty 3)

This is what the FastAPI service will return to Member 4's fusion engine.


In [ ]:
# Extract the k=2 hop neighborhood around the flagged node
subset, sub_ei, mapping, edge_mask = k_hop_subgraph(
    node_idx=top_idx, num_hops=2, edge_index=data.edge_index,
    relabel_nodes=True, num_nodes=data.num_nodes,
)

# Pull data for the subgraph
sub_nodes = subset.tolist()
sub_edge_attrs = data.edge_attr[edge_mask]
sub_attentions = attentions[1][edge_mask]  # second-layer attention weights
sub_y = data.y[subset]
sub_probs = probs[subset]

# Identify the flagged node's position inside the relabeled subgraph
flagged_local = (subset == top_idx).nonzero(as_tuple=True)[0].item()

print(f'Subgraph extracted:')
print(f'  k=2 hop neighborhood of node {top_idx}')
print(f'  {len(sub_nodes):,} nodes  |  {sub_ei.shape[1]:,} edges')
print(f'  Mules in subgraph: {int(sub_y.sum().item())}')


## 6. Visualize the suspicious subgraph

Red node = flagged mule. Edge thickness ∝ Edge-MLP attention weight (Novelty 1).


In [ ]:
# Build a NetworkX graph for visualization (cap nodes shown for clarity)
MAX_NODES = 30
if len(sub_nodes) > MAX_NODES:
    # Pick the highest-attention edges and their endpoints
    top_edges_idx = sub_attentions.argsort(descending=True)[:MAX_NODES]
    keep_edges = sub_ei[:, top_edges_idx]
    keep_nodes = set(keep_edges[0].tolist()) | set(keep_edges[1].tolist())
    keep_nodes.add(flagged_local)
    # Filter edges to only those between kept nodes
    kept_edges_mask = torch.tensor([(int(s) in keep_nodes) and (int(d) in keep_nodes)
                                     for s, d in sub_ei.t()])
    plot_ei = sub_ei[:, kept_edges_mask]
    plot_attn = sub_attentions[kept_edges_mask]
else:
    plot_ei = sub_ei
    plot_attn = sub_attentions
    keep_nodes = set(range(len(sub_nodes)))

G = nx.DiGraph()
for i in keep_nodes:
    real_id = sub_nodes[i]
    G.add_node(i, prob=float(sub_probs[i].item()), is_mule=int(sub_y[i].item()))
for col, (src, dst) in enumerate(plot_ei.t().tolist()):
    G.add_edge(src, dst, weight=float(plot_attn[col].item()))

# Style
node_colors = []
node_sizes = []
for n in G.nodes():
    if n == flagged_local:
        node_colors.append('crimson'); node_sizes.append(900)
    elif G.nodes[n]['is_mule']:
        node_colors.append('orangered'); node_sizes.append(500)
    elif G.nodes[n]['prob'] > 0.5:
        node_colors.append('orange'); node_sizes.append(350)
    else:
        node_colors.append('lightsteelblue'); node_sizes.append(250)

edge_widths = [2 + 6 * G.edges[e]['weight'] for e in G.edges()]
edge_colors = ['crimson' if G.edges[e]['weight'] > 0.7 else 'gray' for e in G.edges()]

fig, ax = plt.subplots(figsize=(13, 9))
pos = nx.spring_layout(G, seed=42, k=1.2)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths,
                       arrows=True, arrowsize=14, ax=ax, alpha=0.7)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ax.set_title(f'Suspicious Subgraph - flagged node {top_idx}\n'
             f'Red center = mule. Edge thickness = Edge-MLP attention. {len(G.nodes())} nodes shown.',
             fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Subgraph visualized - showing top-{MAX_NODES} nodes by edge attention')


## 7. Build the JSON payload that goes to Member 4's Fusion Engine

This is the output Member 4's RAG-grounded LLM consumes for the forensic narrative.


In [ ]:
# Aggregate structural evidence
in_degree_flagged = int((sub_ei[1] == flagged_local).sum().item())
out_degree_flagged = int((sub_ei[0] == flagged_local).sum().item())
fresh_sender_count = sum(
    1 for n in range(len(sub_nodes))
    if int(data.x[sub_nodes[n], 1].item()) <= 1  # out_degree <= 1 -> fresh sender
)
fresh_sender_ratio = fresh_sender_count / max(len(sub_nodes), 1)
mean_drain = float(sub_edge_attrs[:, 1].mean().item())
mules_in_subgraph = int(sub_y.sum().item())

# Build the response
response = {
    'transaction_id': f'TX_DEMO_{top_idx}',
    'model_version': 'graphsage-edge-mlp-focal-v0.3.0',
    'stage': 'stage_3a_focal',
    'relational_risk_score': round(top_prob, 4),
    'risk_level': 'CRITICAL' if top_prob > 0.9 else 'HIGH' if top_prob > 0.5 else 'MEDIUM',
    'confidence': round(max(top_prob, 1 - top_prob), 4),
    'suspicious_subgraph': {
        'k_hop': 2,
        'node_count': len(sub_nodes),
        'edge_count': sub_ei.shape[1],
        'sink_account': f'NODE_{top_idx}',
        'pattern': 'HUB_AND_SPOKE',
        'pattern_confidence': round(min(in_degree_flagged / 5.0, 1.0), 3),
        'structural_evidence': {
            'flagged_in_degree': in_degree_flagged,
            'flagged_out_degree': out_degree_flagged,
            'fresh_sender_ratio': round(fresh_sender_ratio, 3),
            'mean_drain_ratio': round(mean_drain, 3),
            'mules_in_subgraph': mules_in_subgraph,
        },
    },
    'metadata': {
        'tuned_threshold': round(best_thresh, 4),
        'extraction_method': 'k_hop_subgraph_pyg',
    },
}

print(json.dumps(response, indent=2))


## 8. Summary

Live demo proof — for May 11 panel:

1. **Inference works** on Mac (no GPU needed) using the trained Stage 3a checkpoint
2. **Suspicious subgraph extracted** via PyG's `k_hop_subgraph` (Novelty 3)
3. **Edge-MLP attention** (Novelty 1) controls edge thickness in the visualization
4. **Forensic JSON** generated in the locked schema Member 4's RAG engine consumes
5. **End-to-end**: ground-truth fraud node → high model probability → flagged subgraph → JSON in <30 sec

**Test F1: 0.5387  |  AUROC: 0.9497  |  Test Recall: 0.5663**
